# RF-DETR Small refinement

The saved run starts from the best 50-minute checkpoint, with 30% learning rates, 640-pixel input and the same split. Validation occurs each full epoch. Patience is five epochs, minimum cumulative mAP gain 0.002, maximum budget nine hours. Training stopped after 17 epochs; epoch 12 is best.

Default execution reads the completed run; model training is disabled.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import json
import pandas as pd

RUN_TRAINING = False
MODEL = ROOT / 'models/rf-small-refined'
CONFIG = dict(model='rf-small', refine=True, minutes=540., device='gpu', size=640, seed=42,
              train_pages=None, val_pages=None, pilot=False, resume=(MODEL / 'last/state.pt').exists())
if RUN_TRAINING:
    from detectors import train_budget
    train_budget(CONFIG)

record = json.loads((ROOT / 'results/cv/rf-small-refined.json').read_text())['rf-small-refined']
pd.Series(record['training']['config'])

model                              rf-small
minutes                               540.0
size                                    640
seed                                     42
refine                                 True
lr_scale                                0.3
patience                                  5
min_delta                             0.002
init_from    models/benchmark-rf-small/best
dtype: object

In [2]:
history = pd.DataFrame(record['training']['history'])
history['minutes'] = history.seconds / 60
history[['epoch', 'minutes', 'train_loss', 'mAP', 'AP50', 'epoch_complete']].round(4)

,epoch,minutes,train_loss,mAP,AP50,epoch_complete
0,0,0.2326,NaN,0.4251,0.6475,False
1,1,7.3104,4.5217,0.4252,0.6385,True
2,2,14.3595,4.6496,0.4272,0.6556,True
3,3,21.4194,4.1769,0.4364,0.6569,True
4,4,28.4805,4.2772,0.4445,0.6798,True
5,5,35.5327,4.0857,0.4384,0.6855,True
6,6,42.5712,4.0221,0.4644,0.6740,True
7,7,49.6108,4.3197,0.4348,0.6345,True
8,8,56.6400,4.1376,0.4334,0.6676,True
9,9,63.6746,3.8724,0.4438,0.6610,True


In [3]:
best = history.loc[history.mAP.idxmax()]
pd.Series(dict(status=record['training']['status'], best_epoch=int(best.epoch),
               stopped_epoch=int(history.epoch.max()), best_mAP=best.mAP,
               additional_minutes=record['training']['charged_seconds']/60))

status                early_stopped
best_epoch                       12
stopped_epoch                    17
best_mAP                   0.510513
additional_minutes       119.947166
dtype: object

![Refinement learning curve and class AP](../results/cv/rf-small-refined_metrics.png)